### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.
  
#### Weather Agent

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENROUTER_API_KEY"]=os.getenv("OPENROUTER_API_KEY", "")
os.environ["OPENWEATHERMAP_API_KEY"] = os.getenv("OPENWEATHERMAP_API_KEY", "")

In [78]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model = "nvidia/nemotron-3-super-120b-a12b:free")
model

ChatOpenRouter(profile={}, client=<openrouter.sdk.OpenRouter object at 0x0000020178AF55E0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com/oss', app_title='langchain', model_name='nvidia/nemotron-3-super-120b-a12b:free', model_kwargs={})

In [79]:
from langchain.tools import tool
from langchain_community.document_loaders import WeatherDataLoader

@tool
def get_weather(location: str) -> str:
    """"Get the weather at a location"""
    loader = WeatherDataLoader.from_params(
        [location],
        openweathermap_api_key = os.getenv("OPENWEATHERMAP_API_KEY")
    )

    result = loader.load()
    return result[0].page_content

model_with_tools = model.bind_tools([get_weather])

In [80]:
response = model_with_tools.invoke("What's the weather in Gurugram?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user is asking for the weather in Gurugram. I need to use the get_weather tool with location "Gurugram". Let me call that function.\n', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user is asking for the weather in Gurugram. I need to use the get_weather tool with location "Gurugram". Let me call that function.\n', 'format': 'unknown', 'index': 0.0}]} response_metadata={'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'id': 'gen-1774615946-uxbUA6AzDiiRtNvgl7gS', 'created': 1774615946, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019d2f5a-5f49-7e12-88c1-b41f2e441a88-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Gurugram'}, 'id': 'call_613d53754f264dfcbfaa9c4a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 276, 'output_tokens': 61, 'total_tokens': 337, 'input_token_details': {'ca

#### Tool Execution Loops
When a model calls a tool, the tool is executed and the result is returned to the model. This process can be repeated multiple times in a single conversation, allowing the model to iteratively gather information and refine its responses.

In [83]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Gurugram?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

Currently in Gurugram:

*   **Condition:** Haze
*   **Temperature:** 29.0°C (feels like 28.5°C)
*   **Humidity:** 39%
*   **Wind:** 4.6 m/s from the west-northwest (290°)
*   **Cloud cover:** 20%
*   **Rain:** None

The weather is dry and hazy with moderate warmth.


In [ ]:
messages

[{'role': 'user', 'content': "What's the weather in Gurugram?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'I need to get the weather for Gurugram using the get_weather function. The user is asking for the weather in Gurugram, which is a city in India. I\'ll call the function with location set to "Gurugram".\n', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'I need to get the weather for Gurugram using the get_weather function. The user is asking for the weather in Gurugram, which is a city in India. I\'ll call the function with location set to "Gurugram".\n', 'format': 'unknown', 'index': 0.0}]}, response_metadata={'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'id': 'gen-1774615947-T19AZj1ZA9Y3CgkHbxeE', 'created': 1774615947, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019d2f5a-6542-70c2-ae14-344840db925f-0', tool_calls=[{'name': 'get_weather', 'args': {'l